In [2]:
import os
from natsort import natsorted
import random
import torch
import torchvision
from skimage import io
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import color, exposure
from skimage.color import rgb2gray
from scipy.stats import pearsonr
import numpy as np

os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"


userAnDhead_no = 0 # Note that 0,1,2, and 3 indicate Patient 1 Head_1, Patient 1 Head_2, Patient 2 Head_1, etc. 
P_names_list = ['Patient_0001','Patient_0001','Patient_0003','Patient_0003']
H_names_list = ['Head_1','Head_2','Head_1','Head_2']
# no of time based clusters starting from 0...N-1
N = 5

InterClusterCC = []
IntraClusterCC = []

# This method selects the pixels in a radius (half of the width or height) around the center
# ignores the rest of the pixels and also uses RGB info
def masked_corr_coef(img_a, img_b):
    
    img_a = np.array(img_a)
    img_b = np.array(img_b)
    
    width_a = img_a.shape[0]  
    height_a = img_a.shape[1]
    width_b = img_b.shape[0]  
    height_b = img_b.shape[1]
   
    # reduced image does not have pixels outside the defined radius
    reduced_img_a = []
    reduced_img_b = []
    
    if width_a == width_b and width_a == height_a :
          
        # center and radius of the circle
        center_x = width_a//2 # floor divsion (to a whole number)
        center_y = height_a//2
        radius = min(center_x, center_y)
        
        # Iterate through all pixels in the image
        for x in range(width_a):
            for y in range(height_a):
                # Calculate the distance from the center of the circle
                distance = ((x - center_x) ** 2 + (y - center_y) ** 2) ** 0.5

                # Check if the distance is within the circle's radius
                if distance <= radius:
                   # Get the pixel color from the original image
                    pixel_color_a = img_a[x,y,:]
                    pixel_color_b = img_b[x,y,:]
                    reduced_img_a.append(pixel_color_a)
                    reduced_img_b.append(pixel_color_b)
                    
        reduced_img_a = np.array(reduced_img_a)
        reduced_img_b = np.array(reduced_img_b) 
        
        corr_coeff, _ = pearsonr(reduced_img_a.flatten(),reduced_img_b.flatten())
        return corr_coeff
#         print('Size of reduced image',reduced_img_a.shape)
#         print('Size of original image',img_a.shape)
          

def display_images_side_by_side(images_side_by_side):
    num_images = len(images_side_by_side)
    fig, axes = plt.subplots(1, num_images, figsize=(10, 5))
    plt.gray()
    for i in range(num_images):
        image = images_side_by_side[i]
      
        axes[i].imshow(image)
        axes[i].axis('off')    
    plt.tight_layout()
    plt.show()

path_name = r"C:\Users\psh006\OneDrive - UiT Office 365\ML codes\AICE\CorrespondingImages"

# List of folder names for the patients
Pfolder_names = os.listdir(path_name)

head_folder_names = [r"Head_1",r"Head_2"]
SubFolder_names = []
P_names = []
H_names = []
Cluster_names = []
no_of_folders_list = []
for ls in Pfolder_names:
    for sh in head_folder_names:
        
        
        # Join paths and find the subfolders names where data is present
        str = os.path.join(path_name,ls,sh)
        names_of_subFolders = os.listdir(str)
        #print(len(names_of_subFolders))
       
        # Sorted files 
        names_of_subFolders = natsorted(names_of_subFolders) 
        
        max_no_of_folders = len(names_of_subFolders)
        #print(len(names_of_subFolders))
        no_of_folders_list.append(len(names_of_subFolders))
        for i in range(max_no_of_folders):
            
            #print(os.path.join(path_name,ls,sh,names_of_subFolders[i]))
            normalized_f_no= np.int32((i/(max_no_of_folders))*N)
            #print('Normalized f_no',normalized_f_no)
            SubFolder_names.append(os.path.join(path_name,ls,sh,names_of_subFolders[i]))
            Cluster_names.append(normalized_f_no)
            P_names.append(ls)
            H_names.append(sh)

            
print(Cluster_names)
c_nos = range(N) # 0,1,..N-1
img_list = []
cluster_list = []


dims = (no_of_folders_list[userAnDhead_no],256, 256)
#img_arr = np.zeros(256*256*no_of_folders_list[userAnDhead_no]).reshape(dims)
temp_ref = " "
PandHinfo = []
print(no_of_folders_list)

print(no_of_folders_list[userAnDhead_no])
img_list = []

for fc in range(len(SubFolder_names)):
    for cn in c_nos:
        jpeg_list = []
        if Cluster_names[fc] == cn and P_names[fc] == P_names_list[userAnDhead_no]\
        and H_names[fc] == H_names_list[userAnDhead_no]:
            #print('folder name',SubFolder_names[fc])
            # Select Jpeg file from this folder
            #List all files in the folder
            jpeg_list = os.listdir(SubFolder_names[fc]) 
            #print(jpeg_list)
            # selet a jpeg image randomly from this folder
            r_int = random.randint(1, 9)
            full_fname = os.path.join(SubFolder_names[fc],jpeg_list[r_int])
            rgb_img = io.imread(full_fname)
            img_list.append(rgb_img)
            cluster_list.append(cn)
            
img_arr = np.array(img_list)
#print(img_arr.size)
print(len(cluster_list))
count_for_clusters = 0

# Correlation within the same clusters 
for cl_x in range(len(cluster_list)):
    for cl_y in range(len(cluster_list)):
        
        
            
        # if cluster numbers are the same
        if cluster_list[cl_x] == cluster_list[cl_y]:  
            correlation_coefficient_intra  = masked_corr_coef(img_arr[cl_x,:,:,:],img_arr[cl_y,:,:,:])
            IntraClusterCC.append(correlation_coefficient_intra)
            count_for_clusters = count_for_clusters + 1  
        

# Convert the list to a NumPy array
IntraClusterCC_arr = np.array(IntraClusterCC)
print('Mean Correlation Across same time clusters = ',np.mean(IntraClusterCC_arr))
print('Std of Correlations Across same time clusters = ',np.std(IntraClusterCC_arr))
print('Size of IntraClusterCC = ',IntraClusterCC_arr.size)     
## For different clusters select a pair of images to compare
no_of_clusters_to_compare = count_for_clusters # Stopping criterion for inter class comparisons
count_for_clusters = 0

for cl_x in range(len(cluster_list)):
    for cl_y in range(len(cluster_list)):
        
        if count_for_clusters>= no_of_clusters_to_compare:
            break
        # if cluster numbers are different then calculate correlation
        if cluster_list[cl_x] != cluster_list[cl_y] :
            correlation_coefficient_inter = masked_corr_coef(img_arr[cl_x,:,:,:],img_arr[cl_y,:,:,:])
            InterClusterCC.append(correlation_coefficient_inter)
            count_for_clusters = count_for_clusters + 1

           
# Convert the list to a NumPy array
InterClusterCC_arr = np.array(InterClusterCC)
print('Mean Correlation Across different time clusters = ',np.mean(InterClusterCC_arr))
print('Std of Correlations Across different time clusters = ',np.std(InterClusterCC_arr))
print('Size of InterClusterCC = ',InterClusterCC_arr.size)
     
   
        
        
        
        


[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 